In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/naukri_data_science_jobs_india.csv")

In [4]:
df_clean = df.copy()

In [5]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

df_clean.columns

Index(['company', 'location', 'education', 'skills_description', 'job_type',
       'salary', 'job_experience', 'job_role'],
      dtype='str')

In [6]:
# Missing values count
missing_values = df_clean.isnull().sum()

# Missing percentage
missing_percentage = round((missing_values / len(df_clean)) * 100, 2)

# Data Quality Report
missing_report = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percentage
})

missing_report

,Missing Values,Missing %
company,0,0.00
location,0,0.00
education,1676,13.97
skills_description,0,0.00
job_type,1280,10.67
salary,976,8.13
job_experience,0,0.00
job_role,0,0.00


In [7]:
# Fill categorical missing values
df_clean["education"] = df_clean["education"].fillna("Not Specified")
df_clean["job_type"] = df_clean["job_type"].fillna("Not Specified")

# Keep salary as NaN (do not fill)

# Verify
df_clean.isnull().sum()

company                 0
location                0
education               0
skills_description      0
job_type                0
salary                976
job_experience          0
job_role                0
dtype: int64

In [8]:
# Check duplicate rows
duplicate_count = df_clean.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [9]:
# Number of unique locations
print("Unique Locations:", df_clean["location"].nunique())

# Top 30 locations
df_clean["location"].value_counts().head(30)

Unique Locations: 822


location
Bangalore/Bengaluru                                     3383
Pune                                                    1179
Hyderabad/Secunderabad                                   963
Mumbai                                                   816
Gurgaon/Gurugram                                         698
Chennai                                                  660
Bengaluru                                                342
Noida                                                    290
Remote                                                   153
Hyderabad                                                137
Delhi / NCR                                              114
New Delhi                                                111
Kolkata                                                   96
Hyderabad/Secunderabad, Bangalore/Bengaluru               92
Ahmedabad                                                 91
Gurgaon                                                   87
Hyderabad/Secun

In [10]:
# Standardize city names
location_mapping = {
    "Bangalore/Bengaluru": "Bengaluru",
    "Hyderabad/Secunderabad": "Hyderabad",
    "Gurgaon/Gurugram": "Gurugram",
    "Gurgaon": "Gurugram",
    "Delhi / NCR": "Delhi NCR",
    "New Delhi": "Delhi",
    "Kochi/Cochin": "Kochi"
}

for old, new in location_mapping.items():
    df_clean["location"] = df_clean["location"].str.replace(old, new, regex=False)

In [11]:
df_clean["location"].value_counts().head(30)

location
Bengaluru                        3725
Pune                             1179
Hyderabad                        1100
Mumbai                            816
Gurugram                          785
Chennai                           660
Noida                             290
Delhi                             154
Remote                            153
Delhi NCR                         116
Hyderabad, Bengaluru              106
Kolkata                            96
Ahmedabad                          91
Hyderabad, Chennai, Bengaluru      85
Gurugram, Bengaluru                79
Gurugram, Delhi NCR                72
Pune, Bengaluru                    70
Hyderabad, Pune, Bengaluru         66
Chennai, Bengaluru                 65
Kochi                              63
Hyderabad, Chennai                 44
Jaipur                             42
Mohali                             42
Hyderabad, Pune                    37
Chandigarh                         29
Mumbai, Bengaluru                  28
Vad

In [12]:
df_clean["location"].sample(30, random_state=42)

1935               Hyderabad, Chennai, Bengaluru
6494                                    Gurugram
1720                                   Bengaluru
9120                                   Bengaluru
360                                    Bengaluru
9663                                   Bengaluru
5277                                   Bengaluru
8546                                        Pune
2221                                        Pune
4617                                      Nagpur
3465                                       Delhi
1408                                   Bengaluru
468                                    Bengaluru
1175                                   Bengaluru
5224                          Chennai, Bengaluru
8128                                      Mumbai
424      Gurugram, Bengaluru, Mumbai (All Areas)
7047                                  Chandigarh
2483                                   Bengaluru
1825                                   Bengaluru
10557               

In [15]:
sorted(df_clean["location"].unique())[:50]

['Ahmedabad',
 'Ahmedabad(Anand Nagar +7)',
 'Ahmedabad(Ghatlodia), Surat, Vadodara',
 'Ahmedabad(Prahlad Nagar)',
 'Ahmedabad, Ahmedabad',
 'Ahmedabad, Bengaluru',
 'Ahmedabad, Bengaluru, Mumbai (All Areas)',
 'Ahmedabad, Chennai, Bengaluru',
 'Ahmedabad, Gujarat',
 'Ahmedabad, Jaipur, Delhi NCR',
 'Ahmedabad, Mumbai (All Areas)',
 'Ahmedabad, Raipur, ahmedabad,detroj,dholka,viramgam,raipur',
 'Ahmedabad, Rajkot',
 'Ahmedabad, United States (USA)',
 'Alleppey/Alappuzha',
 'Ambah',
 'Amritsar',
 'Anand',
 'Angul, Bellary/Ballari, Bareilly, Anantapur, Hyderabad, Pune, Chennai, Bengaluru, Ankleshwar',
 'Aurangabad, Mumbai (All Areas)',
 'Bahadurgarh',
 'Bangalore Rural',
 'Bangalore Rural, Bengaluru',
 'Bellary/Ballari',
 'Bengaluru',
 'Bengaluru(5th block Koramangala)',
 'Bengaluru(6th Phase JP Nagar)',
 'Bengaluru(BTM Layout)',
 'Bengaluru(Banaswadi)',
 'Bengaluru(Bannerghatta Road)',
 'Bengaluru(Bellandur)',
 'Bengaluru(EPIP Zone +1)',
 'Bengaluru(Electronic City +1)',
 'Bengaluru(Ele

In [16]:
# Remove text inside parentheses
df_clean["location"] = (
    df_clean["location"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [17]:
sorted(df_clean["location"].unique())[:50]

['Ahmedabad',
 'Ahmedabad, Ahmedabad',
 'Ahmedabad, Bengaluru',
 'Ahmedabad, Bengaluru, Mumbai',
 'Ahmedabad, Chennai, Bengaluru',
 'Ahmedabad, Gujarat',
 'Ahmedabad, Jaipur, Delhi NCR',
 'Ahmedabad, Mumbai',
 'Ahmedabad, Raipur, ahmedabad,detroj,dholka,viramgam,raipur',
 'Ahmedabad, Rajkot',
 'Ahmedabad, Surat, Vadodara',
 'Ahmedabad, United States',
 'Alleppey/Alappuzha',
 'Ambah',
 'Amritsar',
 'Anand',
 'Angul, Bellary/Ballari, Bareilly, Anantapur, Hyderabad, Pune, Chennai, Bengaluru, Ankleshwar',
 'Aurangabad, Mumbai',
 'Bahadurgarh',
 'Bangalore Rural',
 'Bangalore Rural, Bengaluru',
 'Bellary/Ballari',
 'Bengaluru',
 'Bengaluru, Bangalore',
 'Bengaluru, Bangalore North',
 'Bengaluru, Belgaum',
 'Bengaluru, Bengaluru / Bangalore',
 'Bengaluru, Bengaluru/Bangalore',
 'Bengaluru, Delhi',
 'Bengaluru, Delhi NCR',
 'Bengaluru, Delhi NCR, Mumbai',
 'Bengaluru, Greater Noida',
 'Bengaluru, India',
 'Bengaluru, Karnataka',
 'Bengaluru, Mumbai',
 'Bengaluru, Mumbai , india',
 'Bengaluru,

In [20]:
# Check unique education values
df_clean["education"].value_counts(dropna=False)

education
UG                1757
Post-graduate     1753
ug                1726
PG                1720
Under-graduate    1685
pg                1683
Not Specified     1676
Name: count, dtype: int64

In [21]:
# Standardize education values
education_mapping = {
    "UG": "Undergraduate",
    "ug": "Undergraduate",
    "Under-graduate": "Undergraduate",
    "PG": "Postgraduate",
    "pg": "Postgraduate",
    "Post-graduate": "Postgraduate"
}

df_clean["education"] = df_clean["education"].replace(education_mapping)

In [22]:
df_clean["education"].value_counts(dropna=False)

education
Undergraduate    5168
Postgraduate     5156
Not Specified    1676
Name: count, dtype: int64

In [23]:
df_clean["job_type"].value_counts(dropna=False)

job_type
contract         5366
full-time        5354
Not Specified    1280
Name: count, dtype: int64

In [24]:
# Basic salary statistics
df_clean["salary"].describe()

count    1.102400e+04
mean     1.410455e+06
std      6.382235e+05
min      3.000220e+05
25%      8.637465e+05
50%      1.414624e+06
75%      1.966901e+06
max      2.499934e+06
Name: salary, dtype: float64

In [27]:
# Check if any salary is zero or negative
print("Zero Salary :", (df_clean["salary"] == 0).sum())
print("Negative Salary :", (df_clean["salary"] < 0).sum())

Zero Salary : 0
Negative Salary : 0


In [28]:
df_clean["job_experience"].value_counts().sort_index()

job_experience
0-1     270
0-2     284
0-3     194
0-4     180
0-5     182
0-6     212
0-7     190
1-2     385
1-3     387
1-4     199
1-5     171
1-6     178
1-7     202
2-3     498
2-4     460
2-5     171
2-6     192
2-7     180
3-4     573
3-5     560
3-6     180
3-7     186
4-5     636
4-6     663
4-7     168
5-6     807
5-7     754
6-7     817
6-8     657
7-8    1464
Name: count, dtype: int64

In [29]:
df_clean["job_experience"].unique()

<StringArray>
['5-6', '1-7', '7-8', '6-7', '3-7', '2-4', '1-4', '3-4', '6-8', '3-5', '4-6',
 '1-3', '4-5', '4-7', '0-1', '0-2', '1-2', '5-7', '2-7', '2-3', '3-6', '0-7',
 '2-5', '1-6', '0-4', '0-6', '1-5', '2-6', '0-3', '0-5']
Length: 30, dtype: str

In [30]:
# Top 30 job roles
df_clean["job_role"].value_counts().head(30)

job_role
Data Engineer                             580
Data Scientist                            505
Data Analyst                              353
Senior Technical Lead (Data Engineer)     276
Senior Data Engineer                      197
Business Analyst                          197
Senior Data Scientist                      97
Azure Data Engineer                        80
Data Engineer: Data Integration            75
Big Data Engineer                          71
Senior Data Analyst                        66
Big Data Developer                         65
Lead Data Engineer                         55
Data Engineer: Big Data                    55
Senior Software Engineer                   51
Senior Business Analyst                    49
Product Analyst                            43
Python Developer                           42
Data Engineering Application Developer     36
Software Engineer                          36
Lead Data Scientist                        35
Analyst                  

In [31]:
# Random sample of job roles
df_clean["job_role"].sample(30, random_state=42)

1935                                          Data Analyst
6494                         Azure Data Engineer - ETL/MDM
1720                                    Sr . Data Engineer
9120                        Business Intelligence Engineer
360                           Data Scientist (Marketplace)
9663                                   Sr Python Developer
5277                  Data Product Engineer, Data Engineer
8546                                           Data Bricks
2221                                  Senior Data Engineer
4617                                          Data Analyst
3465                             CRM Business Data Analyst
1408              KGS - Data Scientist - Assistant Manager
468      Senior Data Scientist/Data Scientist - Researc...
1175                                        Data Scientist
5224                                         Data Engineer
8128                                        Senior Analyst
424             Job | Data Scientist Opportunity with Pa

In [32]:
df_clean["skills_description"].sample(20, random_state=42)

1935    data analysis, communication, analytical, AppD...
6494    Azure, MDM, ETL, DataLake, Data Migration, Clo...
1720    IT Skills, Java, Software Development, Testing...
9120    IT Skills, Software Development, Data Science,...
360     Computer Visions, data warehouse, Julia, deep ...
9663    Database design, Javascript, Agile, Data struc...
5277    Airflow, data architectures, Glue, S3, Github,...
8546    AZURE Database, Azure Databricks, Data Bricks,...
2221    Java, Scala, Apache, Spark, REST APIs, Docker ...
4617    Help Desk, Customer Service, data analysis, cu...
3465    Presentation Skills, Monthly Reports, Excel, S...
1408    Computer science, Logistic regression, deep le...
468     Big data frameworks, Research, Analytics, Scal...
1175    r, Time Series Analysis, Forecasting, arima, p...
5224    Data Engineer, Data aggregation, programming s...
8128    Analyst, digital analytics, Operations researc...
424                                           Python, SQL
7047    code, 

In [33]:
print("Missing Values:", df_clean["skills_description"].isnull().sum())
print("Blank Values:", df_clean["skills_description"].str.strip().eq("").sum())

Missing Values: 0
Blank Values: 0


In [34]:
df_clean.to_csv(
    "../data/cleaned/naukri_data_science_jobs_india_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
